# ML-Bitto — Reproducibility Notebook


[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PLACEHOLDER/ML-Bitto/blob/main/reproduce.ipynb)


This notebook is a publication-oriented, interactive reproducibility companion for ML-Bitto, a PyTorch pipeline for binary classification of clinical and oral images. It demonstrates the full workflow directly in Python: controlled environment setup, synthetic data generation, preprocessing and augmentation, model instantiation, short training, evaluation, per-sample error inspection, Grad-CAM visualization, and optional repeated-holdout cross-validation. The notebook is written so that a reviewer can understand what is being measured and why, without having to inspect the project source code first.


## Table of contents


- [0. Cover](#0.-Cover)
- [1. Environment setup](#1.-Environment-setup)
- [2. Dataset](#2.-Dataset)
- [3. Preprocessing and augmentation](#3.-Preprocessing-and-augmentation)
- [4. Model](#4.-Model)
- [5. Training](#5.-Training)
- [6. Evaluation](#6.-Evaluation)
- [7. Per-sample predictions](#7.-Per-sample-predictions)
- [8. Grad-CAM](#8.-Grad-CAM)
- [9. Cross-validation (optional)](#9.-Cross-validation-(optional))
- [10. Environment and reproducibility report](#10.-Environment-and-reproducibility-report)


| Section | CPU estimate | GPU estimate |
|---|---|---|
| Setup | ~2 min | ~2 min |
| Debug training | ~3 min | ~1 min |
| Evaluation | ~1 min | ~1 min |
| Cross-validation (optional) | ~20 min | ~5 min |


Before publication, replace the `PLACEHOLDER` GitHub URL in this badge and anywhere else it appears in the notebook.

## 1. Environment setup


This section prepares a fully reproducible execution environment. The notebook first detects whether it is running in Google Colab, optionally clones the repository, installs dependencies, and then prints the software stack used for the run. To keep all downstream stochastic operations reproducible, the global random seed is set using the same `set_all_seeds` utility defined in the project entry point. The device summary is printed because hardware affects runtime and, in some settings, numerical behavior.

In [ ]:
%%time
import os
import sys
import json
import math
import random
import subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_NAME = "ML-Bitto"
REPO_URL = "https://github.com/PLACEHOLDER/ML-Bitto.git"

if IN_COLAB:
    if not Path(REPO_NAME).exists():
        subprocess.run(["git", "clone", REPO_URL], check=True)
    os.chdir(REPO_NAME)
elif not Path("requirements.txt").exists() and Path(REPO_NAME, "requirements.txt").exists():
    os.chdir(REPO_NAME)

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-r",
        "requirements.txt",
        "seaborn",
        "ipywidgets",
        "torchinfo",
    ],
    check=True,
)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torchvision
import sklearn

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("seaborn-whitegrid")  # fallback for matplotlib < 3.6

try:
    import seaborn as sns
    HAS_SEABORN = True
except Exception as exc:
    HAS_SEABORN = False
    print(f"Seaborn not available, falling back to matplotlib-only heatmaps: {exc}")

try:
    import ipywidgets as widgets
    from IPython.display import display, Markdown
    HAS_WIDGETS = True
except Exception as exc:
    HAS_WIDGETS = False
    from IPython.display import display, Markdown
    print(f"ipywidgets not available, interactive controls will fall back to static output: {exc}")

try:
    from torchinfo import summary as torchinfo_summary
    HAS_TORCHINFO = True
except Exception as exc:
    HAS_TORCHINFO = False
    print(f"torchinfo not available, using a manual model summary fallback: {exc}")

from main import SyntheticDataset, set_all_seeds

SEED = 42
set_all_seeds(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
versions_df = pd.DataFrame(
    {
        "Package": ["Python", "PyTorch", "Torchvision", "scikit-learn", "NumPy", "Pandas"],
        "Version": [
            sys.version.split()[0],
            torch.__version__,
            torchvision.__version__,
            sklearn.__version__,
            np.__version__,
            pd.__version__,
        ],
    }
)
display(versions_df)

print(f"Device: {device}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    total_mem_gb = props.total_memory / (1024 ** 3)
    print(f"GPU: {props.name}")
    print(f"GPU memory: {total_mem_gb:.2f} GB")
else:
    print("CUDA not available. The notebook will run on CPU.")

## 2. Dataset


To make the notebook executable without any private or clinical data, we use the project's built-in `SyntheticDataset`. This section mirrors the first stages of a paper Methods workflow: define the sample population, inspect representative examples, quantify the class balance, and apply the same train/validation/test ratios used elsewhere in the repository. Even though the data are synthetic, the splitting logic and downstream bookkeeping are the same as for the real pipeline.

In [ ]:
from sklearn.model_selection import train_test_split
from torch.utils.data import Subset
from dataset_split import SplitConfig

synthetic_dataset = SyntheticDataset(n=120, seed=SEED)
split_config = SplitConfig(random_seed=SEED)

labels = np.array(synthetic_dataset._labels)
indices = np.arange(len(synthetic_dataset))

train_val_idx, test_idx = train_test_split(
    indices,
    test_size=split_config.clinical_test_ratio,
    stratify=labels,
    random_state=split_config.random_seed,
)
train_idx, val_idx = train_test_split(
    train_val_idx,
    train_size=split_config.clinical_train_ratio / (split_config.clinical_train_ratio + split_config.clinical_val_ratio),
    stratify=labels[train_val_idx],
    random_state=split_config.random_seed,
)

split_indices = {
    "train": train_idx.tolist(),
    "val": val_idx.tolist(),
    "test": test_idx.tolist(),
}

fig, axes = plt.subplots(4, 4, figsize=(10, 10))
sample_grid_indices = list(range(16))
for ax, idx in zip(axes.flatten(), sample_grid_indices):
    image, label, _ = synthetic_dataset[idx]
    ax.imshow(image)
    ax.set_title(f"Label: {label}")
    ax.axis("off")
fig.suptitle("SyntheticDataset samples (n=120)", fontsize=14)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(5, 4))
class_counts = pd.Series(labels).value_counts().sort_index()
ax.bar(["Negative (0)", "Positive (1)"], class_counts.values, color=["#4C78A8", "#F58518"])
ax.set_title("Class distribution in the synthetic dataset")
ax.set_ylabel("Number of samples")
plt.show()

split_rows = []
for split_name, idxs in split_indices.items():
    split_labels = labels[idxs]
    positives = int(split_labels.sum())
    negatives = int(len(split_labels) - positives)
    split_rows.append(
        {
            "split": split_name,
            "n_samples": len(idxs),
            "n_positive": positives,
            "n_negative": negatives,
            "positive_rate": round(float(positives / len(idxs)), 4),
        }
    )

split_stats_df = pd.DataFrame(split_rows)
display(split_stats_df)
print("Split ratios used:", {
    "train": split_config.clinical_train_ratio,
    "val": split_config.clinical_val_ratio,
    "test": split_config.clinical_test_ratio,
})

## 3. Preprocessing and augmentation


This section makes the data preparation choices explicit. We instantiate the repository's `AugmentationConfig` and `TransformPipeline`, compute normalization statistics on the training subset when feasible, and visualize how conservative augmentations perturb the same underlying image. This is important methodologically: reviewers should be able to verify that preprocessing does not destroy lesion morphology or introduce implausible artifacts.

In [ ]:
from preprocessing import AugmentationConfig, TransformPipeline, compute_dataset_stats, IMAGENET_MEAN, IMAGENET_STD

train_subset_raw = Subset(synthetic_dataset, split_indices["train"])

if len(split_indices["train"]) >= 50:
    try:
        normalization_mean, normalization_std = compute_dataset_stats(train_subset_raw, sample_size=500)
        normalization_note = "Computed from the synthetic training split"
    except Exception as exc:
        normalization_mean, normalization_std = IMAGENET_MEAN, IMAGENET_STD
        normalization_note = f"Fallback to ImageNet statistics because computation failed: {exc}"
else:
    normalization_mean, normalization_std = IMAGENET_MEAN, IMAGENET_STD
    normalization_note = "Fallback to ImageNet statistics because the training split has fewer than 50 images"

aug_config = AugmentationConfig()
transform_pipeline = TransformPipeline(
    augmentation_config=aug_config,
    normalization_mean=normalization_mean,
    normalization_std=normalization_std,
)
transforms_dict = transform_pipeline.get_transforms_dict()

norm_df = pd.DataFrame(
    {
        "channel": ["R", "G", "B"],
        "mean": normalization_mean,
        "std": normalization_std,
    }
)
display(norm_df)
print(f"Normalization choice: {normalization_note}")

print("Train transforms:")
for tr in transforms_dict["train"].transforms:
    print(f"  - {tr}")
print("\nValidation/Test transforms:")
for tr in transforms_dict["val"].transforms:
    print(f"  - {tr}")

sample_indices_for_aug = split_indices["train"][:4]
fig, axes = plt.subplots(len(sample_indices_for_aug), 4, figsize=(12, 3 * len(sample_indices_for_aug)))
if len(sample_indices_for_aug) == 1:
    axes = np.expand_dims(axes, axis=0)

for row_idx, dataset_idx in enumerate(sample_indices_for_aug):
    image, label, _ = synthetic_dataset[dataset_idx]
    axes[row_idx, 0].imshow(image)
    axes[row_idx, 0].set_title(f"Original\nlabel={label}")
    axes[row_idx, 0].axis("off")

    for aug_col in range(1, 4):
        aug_tensor = transforms_dict["train"](image)
        aug_np = aug_tensor.permute(1, 2, 0).cpu().numpy()
        aug_np = aug_np * np.array(normalization_std) + np.array(normalization_mean)
        aug_np = np.clip(aug_np, 0, 1)
        axes[row_idx, aug_col].imshow(aug_np)
        axes[row_idx, aug_col].set_title(f"Augmented #{aug_col}")
        axes[row_idx, aug_col].axis("off")

fig.suptitle("Conservative augmentation examples", fontsize=14)
plt.tight_layout()
plt.show()

## 4. Model


Here we instantiate the publication-facing model class used in the training pipeline: `BinaryClassificationModel` with a ResNet-18 backbone and a custom classifier head. The goal of this section is transparency rather than training speed. We therefore summarize the architecture, count parameters by subsystem, and explicitly report how many parameters are trainable. This helps a reviewer understand the model capacity and the balance between the pretrained backbone and the task-specific head.

In [ ]:
from training_pipeline import BinaryClassificationModel

model = BinaryClassificationModel(model_type="resnet18", pretrained=True, dropout_rate=0.3).to(device)
model.eval()

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params

if HAS_TORCHINFO:
    summary_obj = torchinfo_summary(model, input_size=(1, 3, 224, 224), verbose=0, depth=2)
    summary_rows = []
    for row in summary_obj.summary_list:
        if row.var_name:
            summary_rows.append(
                {
                    "layer": row.var_name,
                    "output_shape": str(row.output_size),
                    "param_count": int(row.num_params),
                }
            )
    summary_df = pd.DataFrame(summary_rows).head(25)
else:
    summary_df = pd.DataFrame(
        [
            {"layer": "backbone", "output_shape": "feature extractor", "param_count": sum(p.numel() for p in model.backbone.parameters()) if hasattr(model, "backbone") else frozen_params},
            {"layer": "head", "output_shape": "2-class logits", "param_count": sum(p.numel() for p in model.head.parameters()) if hasattr(model, "head") else trainable_params},
        ]
    )

display(summary_df)

group_counts = pd.DataFrame(
    {
        "group": ["backbone", "classifier head"],
        "parameters": [
            sum(p.numel() for p in model.backbone.parameters()) if hasattr(model, "backbone") else frozen_params,
            sum(p.numel() for p in model.head.parameters()) if hasattr(model, "head") else trainable_params,
        ],
    }
)

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(group_counts["group"], group_counts["parameters"], color=["#4C78A8", "#F58518"])
ax.set_title("Parameter count by model group")
ax.set_ylabel("Number of parameters")
plt.show()

param_report_df = pd.DataFrame(
    {
        "quantity": ["total", "trainable", "frozen"],
        "parameters": [total_params, trainable_params, frozen_params],
    }
)
display(param_report_df)

## 5. Training


This section runs a short but real training loop directly from the core training utilities, rather than calling the project entry point as a black box. The objective is not to maximize performance on synthetic data, but to expose the optimization behavior, show which metrics are tracked, and make the learning dynamics inspectable. We train for three epochs, collect the main validation metrics after each epoch, and annotate the best epoch in the resulting plots.

In [ ]:
%%time

from torch.utils.data import DataLoader

from preprocessing import TransformedDataset

from training_pipeline import train_epoch, validate_epoch, compute_class_weights



USE_CLASS_WEIGHTS = True

if HAS_WIDGETS:

    print("A class-weight checkbox can be edited before rerunning this cell if you want to compare weighted vs unweighted loss.")

    display(widgets.Checkbox(value=True, description="Use class-weighted loss (rerun cell after changing)"))



train_dataset = TransformedDataset(Subset(synthetic_dataset, split_indices["train"]), transforms_dict["train"])

val_dataset = TransformedDataset(Subset(synthetic_dataset, split_indices["val"]), transforms_dict["val"])

test_dataset = TransformedDataset(Subset(synthetic_dataset, split_indices["test"]), transforms_dict["test"])



batch_size = 16

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)

val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)



model = BinaryClassificationModel(model_type="resnet18", pretrained=True, dropout_rate=0.3).to(device)

if USE_CLASS_WEIGHTS:

    criterion = torch.nn.CrossEntropyLoss(weight=compute_class_weights(train_loader, str(device)))

else:

    criterion = torch.nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)



history_rows = []

best_val_loss = float("inf")

best_metric_epoch = 0

best_metric_value = -1.0

monitor_metric = "recall"



for epoch in range(1, 4):

    train_metrics = train_epoch(model, train_loader, criterion, optimizer, str(device))

    val_metrics = validate_epoch(model, val_loader, criterion, str(device))

    history_rows.append(

        {

            "epoch": epoch,

            "train_loss": train_metrics["loss"],

            "train_acc": train_metrics["accuracy"],

            "val_loss": val_metrics["loss"],

            "val_acc": val_metrics["accuracy"],

            "val_recall": val_metrics.get("recall", val_metrics.get("sensitivity", 0.0)),

            "val_f1": val_metrics["f1"],

        }

    )

    if val_metrics["loss"] < best_val_loss:

        best_val_loss = val_metrics["loss"]

    _monitored_val = val_metrics.get(monitor_metric) or val_metrics.get("sensitivity") or 0.0
    if _monitored_val > best_metric_value:

        best_metric_value = _monitored_val

        best_metric_epoch = epoch

    print(

        f"Epoch {epoch}: train_loss={train_metrics['loss']:.4f}, train_acc={train_metrics['accuracy']:.4f}, "

        f"val_loss={val_metrics['loss']:.4f}, val_acc={val_metrics['accuracy']:.4f}, "

        f"val_recall={val_metrics['recall']:.4f}, val_f1={val_metrics['f1']:.4f}"

    )



history_df = pd.DataFrame(history_rows)

display(history_df)



best_loss_epoch = int(history_df.loc[history_df["val_loss"].idxmin(), "epoch"])



fig, axes = plt.subplots(1, 2, figsize=(14, 5))



axes[0].plot(history_df["epoch"], history_df["train_loss"], marker="o", label="Train loss")

axes[0].plot(history_df["epoch"], history_df["val_loss"], marker="o", label="Val loss")

axes[0].axvline(best_loss_epoch, color="#E45756", linestyle="--", label=f"Best val loss (epoch {best_loss_epoch})")

axes[0].set_title("Optimization curves")

axes[0].set_xlabel("Epoch")

axes[0].set_ylabel("Loss")

axes[0].legend()



axes[1].plot(history_df["epoch"], history_df["val_acc"], marker="o", label="Val accuracy")

axes[1].plot(history_df["epoch"], history_df["val_recall"], marker="o", label="Val recall")

axes[1].plot(history_df["epoch"], history_df["val_f1"], marker="o", label="Val F1")

axes[1].axvline(best_metric_epoch, color="#54A24B", linestyle="--", label=f"Best recall (epoch {best_metric_epoch})")

axes[1].set_title("Validation metrics across epochs")

axes[1].set_xlabel("Epoch")

axes[1].set_ylabel("Metric value")

axes[1].legend()



plt.tight_layout()

plt.show()

## 6. Evaluation


The evaluation stage is performed directly from the reusable project functions so that every result can be inspected inline. We compute probabilities on the held-out test split, derive the standard binary classification metrics used in the project, and visualize discrimination, calibration, threshold behavior, and score distributions. This mirrors the reporting expectations of a journal supplementary notebook: the numerical outputs and the diagnostic plots are both visible in the same place.

In [ ]:
%%time

from sklearn.metrics import (

    accuracy_score,

    precision_score,

    recall_score,

    f1_score,

    roc_auc_score,

    cohen_kappa_score,

    brier_score_loss,

    roc_curve,

    precision_recall_curve,

    confusion_matrix,

    auc as sklearn_auc,

)

from sklearn.calibration import calibration_curve

from evaluation import run_inference, EvaluationConfig



eval_config = EvaluationConfig(device=str(device), threshold=0.5, output_dir="evaluation_results_notebook")

probs, pred_labels, true_labels, metadata_list = run_inference(model, test_loader, eval_config)



metrics_table = pd.DataFrame(

    {

        "metric": ["accuracy", "precision", "recall", "f1", "auc", "brier_score", "cohen_kappa"],

        "value": [

            accuracy_score(true_labels, pred_labels),

            precision_score(true_labels, pred_labels, zero_division=0),

            recall_score(true_labels, pred_labels, zero_division=0),

            f1_score(true_labels, pred_labels, zero_division=0),

            roc_auc_score(true_labels, probs) if len(np.unique(true_labels)) > 1 else np.nan,

            brier_score_loss(true_labels, probs),

            cohen_kappa_score(true_labels, pred_labels),

        ],

    }

)



def metric_color(value):

    if pd.isna(value):

        return "background-color: #f0f0f0"

    if value > 0.7:

        return "background-color: #c7f9cc"

    if value >= 0.5:

        return "background-color: #fff3bf"

    return "background-color: #ffc9c9"



try:
    display(metrics_table.style.format({"value": "{:.4f}"}).map(metric_color, subset=["value"]))
except AttributeError:
    display(metrics_table.style.format({"value": "{:.4f}"}).applymap(metric_color, subset=["value"]))



fpr, tpr, roc_thresholds = roc_curve(true_labels, probs)

roc_auc_value = roc_auc_score(true_labels, probs) if len(np.unique(true_labels)) > 1 else np.nan

prec_curve, rec_curve, pr_thresholds = precision_recall_curve(true_labels, probs)

pr_auc_value = sklearn_auc(rec_curve, prec_curve)

current_precision = precision_score(true_labels, pred_labels, zero_division=0)

current_recall = recall_score(true_labels, pred_labels, zero_division=0)

cm = confusion_matrix(true_labels, pred_labels, labels=[0, 1])

cm_percent = cm / cm.sum(axis=1, keepdims=True)

calib_true, calib_pred = calibration_curve(true_labels, probs, n_bins=8, strategy="uniform")



fig, axes = plt.subplots(2, 3, figsize=(18, 10))



axes[0, 0].plot(fpr, tpr, color="#4C78A8", linewidth=2, label=f"AUC = {roc_auc_value:.3f}")

axes[0, 0].fill_between(fpr, tpr, alpha=0.2, color="#4C78A8")

axes[0, 0].plot([0, 1], [0, 1], linestyle="--", color="black", alpha=0.6)

axes[0, 0].set_title("ROC curve")

axes[0, 0].set_xlabel("False positive rate")

axes[0, 0].set_ylabel("True positive rate")

axes[0, 0].legend()



axes[0, 1].plot(rec_curve, prec_curve, color="#F58518", linewidth=2, label=f"PR AUC = {pr_auc_value:.3f}")

axes[0, 1].scatter([current_recall], [current_precision], color="#E45756", s=80, label="Operating point")

axes[0, 1].set_title("Precision-Recall curve")

axes[0, 1].set_xlabel("Recall")

axes[0, 1].set_ylabel("Precision")

axes[0, 1].legend()



if HAS_SEABORN:

    annot = np.array([[f"{cm[i, j]}\n({cm_percent[i, j]:.1%})" for j in range(2)] for i in range(2)])

    sns.heatmap(cm, annot=annot, fmt="", cmap="Blues", cbar=False, ax=axes[0, 2])

else:

    im = axes[0, 2].imshow(cm, cmap="Blues")

    for i in range(2):

        for j in range(2):

            axes[0, 2].text(j, i, f"{cm[i, j]}\n({cm_percent[i, j]:.1%})", ha="center", va="center")

axes[0, 2].set_title("Confusion matrix")

axes[0, 2].set_xlabel("Predicted label")

axes[0, 2].set_ylabel("True label")

axes[0, 2].set_xticklabels(["Negative", "Positive"])

axes[0, 2].set_yticklabels(["Negative", "Positive"])



axes[1, 0].plot([0, 1], [0, 1], linestyle="--", color="black", alpha=0.6)

axes[1, 0].plot(calib_pred, calib_true, marker="o", color="#54A24B", label=f"Brier = {brier_score_loss(true_labels, probs):.3f}")

axes[1, 0].set_title("Calibration curve")

axes[1, 0].set_xlabel("Mean predicted probability")

axes[1, 0].set_ylabel("Fraction of positives")

axes[1, 0].legend()



axes[1, 1].hist(probs[true_labels == 0], bins=10, alpha=0.7, label="True negative class", color="#4C78A8")

axes[1, 1].hist(probs[true_labels == 1], bins=10, alpha=0.7, label="True positive class", color="#E45756")

axes[1, 1].axvline(eval_config.threshold, color="black", linestyle="--", label=f"Threshold = {eval_config.threshold:.2f}")

axes[1, 1].set_title("Predicted score distribution")

axes[1, 1].set_xlabel("Predicted probability")

axes[1, 1].set_ylabel("Count")

axes[1, 1].legend()



axes[1, 2].axis("off")

axes[1, 2].text(

    0.0,

    0.95,

    "Evaluation summary\n"

    f"n_test = {len(true_labels)}\n"

    f"threshold = {eval_config.threshold:.2f}\n"

    f"accuracy = {accuracy_score(true_labels, pred_labels):.3f}\n"

    f"precision = {current_precision:.3f}\n"

    f"recall = {current_recall:.3f}\n"

    f"f1 = {f1_score(true_labels, pred_labels, zero_division=0):.3f}",

    va="top",

    fontsize=12,

)



plt.tight_layout()

plt.show()



def render_threshold_view(threshold):

    threshold_preds = (probs >= threshold).astype(int)

    threshold_cm = confusion_matrix(true_labels, threshold_preds, labels=[0, 1])

    threshold_metrics = pd.DataFrame(

        {

            "metric": ["accuracy", "precision", "recall", "f1"],

            "value": [

                accuracy_score(true_labels, threshold_preds),

                precision_score(true_labels, threshold_preds, zero_division=0),

                recall_score(true_labels, threshold_preds, zero_division=0),

                f1_score(true_labels, threshold_preds, zero_division=0),

            ],

        }

    )

    try:
        display(threshold_metrics.style.format({"value": "{:.4f}"}).map(metric_color, subset=["value"]))
    except AttributeError:
        display(threshold_metrics.style.format({"value": "{:.4f}"}).applymap(metric_color, subset=["value"]))

    fig, ax = plt.subplots(figsize=(4, 4))

    if HAS_SEABORN:

        annot = np.array([[str(v) for v in row] for row in threshold_cm])

        sns.heatmap(threshold_cm, annot=annot, fmt="", cmap="Blues", cbar=False, ax=ax)

    else:

        ax.imshow(threshold_cm, cmap="Blues")

        for i in range(2):

            for j in range(2):

                ax.text(j, i, str(threshold_cm[i, j]), ha="center", va="center")

    ax.set_title(f"Confusion matrix @ threshold {threshold:.2f}")

    ax.set_xlabel("Predicted label")

    ax.set_ylabel("True label")

    plt.show()



if HAS_WIDGETS:

    threshold_slider = widgets.FloatSlider(value=0.5, min=0.0, max=1.0, step=0.01, description="Threshold")

    display(widgets.interactive_output(render_threshold_view, {"threshold": threshold_slider}), threshold_slider)

else:

    print("ipywidgets not available; showing a static threshold view at 0.50.")

    render_threshold_view(0.5)

## 7. Per-sample predictions


Aggregate metrics can hide systematic errors, so this section moves from summary statistics to individual predictions. We assemble a prediction table for the full test split, highlight errors, and then visualize the most confidently wrong cases. This is the kind of qualitative audit that is often helpful in supplementary material because it exposes failure modes directly.

In [ ]:
from IPython.display import HTML



test_metadata_df = pd.DataFrame(metadata_list)

predictions_df = pd.DataFrame(

    {

        "image_id": test_metadata_df.get("image_id", pd.Series(range(len(true_labels)))).astype(str),

        "true_label": true_labels,

        "predicted_label": pred_labels,

        "confidence": probs,

        "correct": pred_labels == true_labels,

    }

).sort_values(by="confidence", ascending=False).reset_index(drop=True)



def highlight_errors(row):

    return ["background-color: #ffc9c9" if not row["correct"] else "" for _ in row]



display(predictions_df.style.format({"confidence": "{:.4f}"}).apply(highlight_errors, axis=1))



wrong_df = predictions_df.loc[~predictions_df["correct"]].copy()

false_pos = wrong_df.loc[(wrong_df["true_label"] == 0) & (wrong_df["predicted_label"] == 1)].nlargest(4, "confidence")

false_neg = wrong_df.loc[(wrong_df["true_label"] == 1) & (wrong_df["predicted_label"] == 0)].nsmallest(4, "confidence")

selected_wrong = pd.concat([false_pos, false_neg], axis=0).head(8)



fig, axes = plt.subplots(2, 4, figsize=(16, 8))

axes = axes.flatten()

for ax in axes:

    ax.axis("off")



metadata_lookup = {str(meta.get("image_id", idx)): idx for idx, meta in enumerate(metadata_list)}



for ax, (_, row) in zip(axes, selected_wrong.iterrows()):

    idx_in_test = metadata_lookup.get(str(row["image_id"]))

    if idx_in_test is None:

        ax.set_title("Sample not found")

        continue

    image_tensor, _, _ = test_dataset[idx_in_test]

    image_np = image_tensor.permute(1, 2, 0).cpu().numpy()

    image_np = image_np * np.array(normalization_std) + np.array(normalization_mean)

    image_np = np.clip(image_np, 0, 1)

    ax.imshow(image_np)

    ax.set_title(

        f"ID {row['image_id']}\ntrue={row['true_label']} pred={row['predicted_label']}\nconf={row['confidence']:.3f}",

        fontsize=10,

    )

    ax.axis("off")



fig.suptitle("Most confident misclassifications (up to 4 FP + 4 FN)", fontsize=14)

plt.tight_layout()

plt.show()

## 8. Grad-CAM


Grad-CAM highlights which spatial regions contribute most strongly to the model's decision for a selected class. In a clinical context, the heatmap should ideally focus on lesion structure and immediately adjacent tissue rather than on irrelevant background regions. This does not prove causal reasoning, but it provides a qualitative check that the classifier attends to visually plausible regions. The section below shows both correctly classified and misclassified cases so that attention patterns can be compared.

In [ ]:
%%time

from gradcam import get_target_layer, GradCAM, generate_gradcam

# Safety: metadata_lookup is built in cell 14; redefine here if that cell was skipped.
if "metadata_lookup" not in vars():
    metadata_lookup = {str(meta.get("image_id", idx)): idx for idx, meta in enumerate(metadata_list)}



target_layer = get_target_layer(model, model_type="resnet18")



correct_cases = predictions_df.loc[predictions_df["correct"]].head(3)

wrong_cases = predictions_df.loc[~predictions_df["correct"]].head(3)

gradcam_cases = pd.concat([correct_cases, wrong_cases], axis=0).reset_index(drop=True)



def tensor_to_display_image(image_tensor):

    image_np = image_tensor.permute(1, 2, 0).cpu().numpy()

    image_np = image_np * np.array(normalization_std) + np.array(normalization_mean)

    return np.clip(image_np, 0, 1)



def render_gradcam_case(case_row):

    idx_in_test = metadata_lookup.get(str(case_row["image_id"]))

    image_tensor, _, _ = test_dataset[idx_in_test]

    heatmap, overlay = generate_gradcam(model, image_tensor.unsqueeze(0).to(device), target_layer)

    original = tensor_to_display_image(image_tensor)

    fig, axes = plt.subplots(1, 2, figsize=(8, 4))

    axes[0].imshow(original)

    axes[0].set_title("Original image")

    axes[0].axis("off")

    axes[1].imshow(overlay)

    axes[1].set_title(

        f"Grad-CAM overlay\ntrue={case_row['true_label']} pred={case_row['predicted_label']} conf={case_row['confidence']:.3f}"

    )

    axes[1].axis("off")

    plt.tight_layout()

    plt.show()



fig, axes = plt.subplots(len(gradcam_cases), 2, figsize=(10, 4 * max(len(gradcam_cases), 1)))

if len(gradcam_cases) == 1:

    axes = np.expand_dims(axes, axis=0)



for row_idx, (_, case_row) in enumerate(gradcam_cases.iterrows()):

    idx_in_test = metadata_lookup.get(str(case_row["image_id"]))

    image_tensor, _, _ = test_dataset[idx_in_test]

    _, overlay = generate_gradcam(model, image_tensor.unsqueeze(0).to(device), target_layer)

    original = tensor_to_display_image(image_tensor)

    axes[row_idx, 0].imshow(original)

    axes[row_idx, 0].set_title(f"Original\nID={case_row['image_id']}")

    axes[row_idx, 0].axis("off")

    axes[row_idx, 1].imshow(overlay)

    axes[row_idx, 1].set_title(

        f"Grad-CAM\ntrue={case_row['true_label']} pred={case_row['predicted_label']} conf={case_row['confidence']:.3f}"

    )

    axes[row_idx, 1].axis("off")



fig.suptitle("Grad-CAM examples: correct and incorrect predictions", fontsize=14)

plt.tight_layout()

plt.show()



if HAS_WIDGETS and len(gradcam_cases) > 0:

    options = [

        (f"ID {row['image_id']} | true={row['true_label']} pred={row['predicted_label']}", idx)

        for idx, row in gradcam_cases.iterrows()

    ]

    dropdown = widgets.Dropdown(options=options, description="Sample")



    def _show_gradcam(selected_idx):

        render_gradcam_case(gradcam_cases.iloc[selected_idx])



    display(widgets.interactive_output(_show_gradcam, {"selected_idx": dropdown}), dropdown)

else:

    print("ipywidgets not available; interactive Grad-CAM selector skipped.")

## 9. Cross-validation (optional)


Repeated holdout is used here instead of standard k-fold because the repository is oriented toward small clinical-style datasets where preserving an explicitly held-out test partition per repeat can be more interpretable than rotating every sample through every fold. In this notebook the cross-validation block is optional and shortened to three repeats for reviewer convenience.

In [ ]:
RUN_CROSS_VAL = False  # Set to True to run repeated holdout cross-validation

In [ ]:
%%time
from cross_validation import run_repeated_holdout
from data_loader import ImageMetadata
from sklearn.model_selection import train_test_split

class SyntheticClinicalAdapter(torch.utils.data.Dataset):
    def __init__(self, base_dataset):
        self.base_dataset = base_dataset
        self.samples = []
        for idx in range(len(base_dataset)):
            _, label, meta = base_dataset[idx]
            self.samples.append(
                (
                    meta.get("image_name", f"synthetic_{idx}.png"),
                    ImageMetadata(
                        image_id=str(meta.get("image_id", idx)),
                        image_name=meta.get("image_name", f"synthetic_{idx}.png"),
                        label=int(label),
                        dataset_source="SYNTHETIC_CLINICAL",
                        patient_id=f"patient_{idx:04d}",
                    ),
                )
            )
    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        image, label, meta = self.base_dataset[idx]
        meta = dict(meta)
        meta["patient_id"] = f"patient_{idx:04d}"
        return image, label, meta

if RUN_CROSS_VAL:
    synthetic_clinical_dataset = SyntheticClinicalAdapter(synthetic_dataset)
    def train_eval_fn(cv_loaders):
        cv_model = BinaryClassificationModel(model_type="resnet18", pretrained=True, dropout_rate=0.3).to(device)
        cv_criterion = torch.nn.CrossEntropyLoss(weight=compute_class_weights(cv_loaders["train"], str(device)))
        cv_optimizer = torch.optim.Adam(cv_model.parameters(), lr=1e-4, weight_decay=1e-4)
        for _ in range(3):
            train_epoch(cv_model, cv_loaders["train"], cv_criterion, cv_optimizer, str(device))
        cv_metrics = validate_epoch(cv_model, cv_loaders["test"], cv_criterion, str(device))

        return {
            "accuracy": cv_metrics.get("accuracy"),
            "sensitivity": cv_metrics.get("recall"),
            "f1": cv_metrics.get("f1"),
            "auc": cv_metrics.get("auc"),
        }

    cv_results = run_repeated_holdout(
        dataset=synthetic_clinical_dataset,
        train_eval_fn=train_eval_fn,
        n_repeats=3,
        batch_size=16,
        num_workers=0,
    )

    cv_summary_df = pd.DataFrame.from_dict(cv_results["summary"], orient="index").reset_index().rename(columns={"index": "metric"})
    display(cv_summary_df[["metric", "mean", "std", "ci_lower", "ci_upper"]])

    plot_df = []
    for metric in ["accuracy", "sensitivity", "f1", "auc"]:
        values = cv_results.get("summary", {}).get(metric, {}).get("values", []) if isinstance(cv_results, dict) else []
        for value in values:
            plot_df.append({"metric": metric, "value": value})
    plot_df = pd.DataFrame(plot_df)

    fig, ax = plt.subplots(figsize=(8, 5))

    if HAS_SEABORN and not plot_df.empty:
        sns.boxplot(data=plot_df, x="metric", y="value", ax=ax)
    else:
        grouped = [plot_df.loc[plot_df["metric"] == m, "value"].values for m in ["accuracy", "sensitivity", "f1", "auc"]]
        ax.boxplot(grouped, labels=["accuracy", "sensitivity", "f1", "auc"])
    ax.set_title("Repeated holdout metric distribution (n=3 repeats)")
    ax.set_xlabel("Metric")
    ax.set_ylabel("Value")
    plt.show()
else:
    print("Cross-validation skipped. Set RUN_CROSS_VAL = True and rerun this cell to execute repeated holdout.")

## 10. Environment and reproducibility report


The final section consolidates the computational context of the notebook run. In publication settings this is important because identical code can behave differently across software versions and hardware backends. The report below records versions, seed, device, and the principal hyperparameters used in this notebook instance.

In [ ]:
reproduce_config = {
    "seed": SEED,
    "batch_size": batch_size,
    "epochs": 3,
    "model_type": "resnet18",
    "dropout": 0.3,
    "split_ratios": {
        "train": split_config.clinical_train_ratio,
        "val": split_config.clinical_val_ratio,
        "test": split_config.clinical_test_ratio,
    },
    "normalization_stats": {
        "mean": normalization_mean,
        "std": normalization_std,
        "note": normalization_note,
    },
    "monitor_metric": monitor_metric,
}

environment_report_df = pd.DataFrame(
    {
        "field": [
            "Python",
            "PyTorch",
            "Torchvision",
            "scikit-learn",
            "NumPy",
            "Pandas",
            "seed",
            "device",
        ],
        "value": [
            sys.version.split()[0],
            torch.__version__,
            torchvision.__version__,
            sklearn.__version__,
            np.__version__,
            pd.__version__,
            SEED,
            str(device),
        ],
    }
)

display(environment_report_df)
print(json.dumps(reproduce_config, indent=2))

## Final checklist


- [x] Environment setup verified
- [x] Dataset loaded and visualized
- [x] Preprocessing pipeline verified
- [x] Model instantiated
- [x] Training completed
- [x] Evaluation metrics computed
- [x] Grad-CAM visualizations generated
- [ ] Cross-validation (optional — set `RUN_CROSS_VAL = True`)